# 呵，又是装东西的一天

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pip

!pip install neurodsp

In [ ]:
!pip install fooof

In [ ]:
!pip install specparam

In [ ]:
import specparam
print(specparam.__version__)

In [ ]:
import specparam
specparam.__version__

# 按cond分epoch，大分特分

In [ ]:
import mne
from pathlib import Path
import os

data_dir = Path("F:/cue_epoch")

cue1_keys = ["cue/ext/rep/cue1","cue/ext/swi/cue1","cue/int/rep/cue1","cue/int/swi/cue1"]
cuesat_keys = ["cue/ext/rep/sat","cue/ext/swi/sat","cue/int/rep/sat","cue/int/swi/sat"]

for ep_fif in data_dir.glob("*-epo.fif"):
    epochs = mne.read_epochs(ep_fif, preload=True)

    cue1_list = [epochs[k] for k in cue1_keys if k in epochs.event_id]
    cuesat_list = [epochs[k] for k in cuesat_keys if k in epochs.event_id]

    if cue1_list:
        cue1_epochs = mne.concatenate_epochs(cue1_list)
        cue1_epochs.save(ep_fif.with_name(ep_fif.stem + "_cue1-epo.fif"), overwrite=True)
    else:
        print(f"[SKIP cue1] {ep_fif.name}: none of {cue1_keys} found in event_id")

    if cuesat_list:
        cuesat_epochs = mne.concatenate_epochs(cuesat_list)
        cuesat_epochs.save(ep_fif.with_name(ep_fif.stem + "_cuesat-epo.fif"), overwrite=True)
    else:
        print(f"[SKIP cuesat] {ep_fif.name}: none of {cuesat_keys} found in event_id")

In [ ]:
data_dir = Path("F:/cue1_epoch")
epo_files = sorted(data_dir.glob("sub_*_cue-epo_cue1-epo.fif"))
for f in epo_files:
    ep = mne.read_epochs(f, preload=True, verbose="ERROR").copy().pick_types(eeg=True)
    print(f.name, len(ep.ch_names))

# Cue1

In [ ]:
#按Cond Compute power spectrum - cue1
import re
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path 
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

data_dir = Path("F:/cue1_epoch")
epo_files = sorted(data_dir.glob("sub_*_cue-epo_cue1-epo.fif"))
conds = ["cue/ext/rep/cue1","cue/ext/swi/cue1","cue/int/rep/cue1","cue/int/swi/cue1"]
out_path = data_dir/"psd_4conds_subjectlevel_eegonly.npz"

#PSD
fmin,fmax=4.0, 30.0
method="welch"
n_fft=256
n_overlap=100

def parse_subj_id(fname: str) -> str:
    m = re.match(r"(sub_[^_]+_\d+_\d+)_cue-epo_cue1-epo\.fif$", fname)
    return m.group(1) if m else Path(fname).stem

subj_ids = []
tmp_store = {c: [] for c in conds}
freqs_ref, ch_names_ref = None, None

epochs0 = mne.read_epochs(epo_files[0], preload=True, verbose="ERROR").pick("eeg")
ch_names_ref = epochs0.ch_names
n_ch = len(ch_names_ref)

for f in epo_files:
    subj_ids.append(parse_subj_id(f.name))
    epochs = mne.read_epochs(f, preload=True, verbose="ERROR")
    epochs = epochs.copy().pick_types(eeg=True)


    for cond in conds:
        if cond not in epochs.event_id:
            tmp_store[cond].append(None)
            continue

        ep = epochs[cond]
        psd = ep.compute_psd(
            method=method, fmin=fmin, fmax=fmax,
            n_fft=n_fft, n_overlap=n_overlap,
            verbose="ERROR",
        )
        psd_data = psd.get_data()   # (n_trials, n_ch, n_freq)
        freqs = psd.freqs

        if freqs_ref is None:
            freqs_ref = freqs
            n_freq = len(freqs_ref)
        else:
            if len(freqs_ref) != len(freqs) or not np.allclose(freqs_ref, freqs):
                raise ValueError("freqs 不一致：请统一采样率/n_fft/fmin-fmax")

        subj_mean = psd_data.mean(axis=0)  # (n_ch, n_freq)
        tmp_store[cond].append(subj_mean)

# cond -> (n_subj, n_ch, n_freq)
psd_stack = {}
for cond in conds:
    arr = np.full((len(subj_ids), n_ch, len(freqs_ref)), np.nan, dtype=float)
    for i, v in enumerate(tmp_store[cond]):
        if v is not None:
            arr[i] = v
    psd_stack[cond] = arr

#plot
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
axes = axes.ravel()

for ax, cond in zip(axes, conds):
    arr = psd_stack[cond]                       # (n_subj, n_ch, n_freq)
    grand = np.nanmean(arr, axis=(0, 1))        # 平均 subjects + channels -> (n_freq,)
    ax.plot(freqs_ref, grand)
    ax.set_title(cond)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power (V²/Hz)")
    ax.grid(True, alpha=0.3)
    miss = np.isnan(arr).all(axis=(1, 2)).sum()
    if miss > 0:
        ax.text(0.02, 0.95, f"missing subj: {miss}", transform=ax.transAxes, va="top")

plt.tight_layout()
plt.show()

In [ ]:
out_path = Path("F:/cue1_epoch/psd_4conds_subjectlevel_eegonly.npz")

np.savez_compressed(
    out_path,
    subj_ids=np.array(subj_ids, dtype=object),
    ch_names=np.array(ch_names_ref, dtype=object),
    freqs=freqs_ref,
    **psd_stack,
)

print("Saved to:", out_path)

In [ ]:
#分离1/f背景 - Cue1
import specparam
import numpy as np
import matplotlib.pyplot as plt
from specparam import SpectralModel
import pandas as pd

conds = ["cue/ext/rep/cue1","cue/ext/swi/cue1","cue/int/rep/cue1","cue/int/swi/cue1"]
npz_path = Path("F:/cue1_epoch/psd_4conds_subjectlevel_eegonly.npz")
d = np.load(npz_path, allow_pickle=True)
freqs_ref = d["freqs"]
freqs = freqs_ref
f_fit = (4, 30)

roi_map = {
    "frontal":   ["Fp1","Fp2","AF3","AF4","F7","F3","Fz","F4","F8","FC5","FC1","FC2","FC6"],
    "parietal":  ["CP5","CP1","CP2","CP6","P7","P3","Pz","P4","P8","PO3","PO4"],
    "occipital": ["O1","Oz","O2","PO7","PO8"],
}


roi_idx = {}
for roi_name, chs in roi_map.items():
    idx = [ch_names_ref.index(ch) for ch in chs if ch in ch_names_ref]
    if len(idx) == 0:
        raise ValueError(f"{roi_name} ROI 在 ch_names_ref 里一个都没匹配到，检查通道命名啊！！！！！")
    roi_idx[roi_name] = idx

#specparam
def pick_best_peak(peaks, band):
    #试啊
    for cf_key, pw_key, bw_key in [("cf","pw","bw"), ("center_frequency","power","bandwidth")]:
        try:
            cf = np.array(sm.get_params("periodic", cf_key), dtype=float)
            pw = np.array(sm.get_params("periodic", pw_key), dtype=float)
            bw = np.array(sm.get_params("periodic", bw_key), dtype=float)
            break
        except Exception:
            cf = pw = bw = None
    if cf is None or cf.size == 0:
        return (np.nan, np.nan, np.nan)
    fmin, fmax = band
    mask = (cf >= fmin) & (cf <= fmax)
    if not np.any(mask):
        return (np.nan, np.nan, np.nan)

    sel = np.where(mask)[0][np.argmax(pw[mask])]
    return (cf[sel], pw[sel], bw[sel])

# 需要检测的频段
target_band = (4, 15)  

rows = []

for si, sid in enumerate(subj_ids):
    for cond in conds:
        arr = psd_stack[cond]  # (n_subj, n_ch, n_freq)
        if np.isnan(arr[si]).all():
            for roi in roi_idx:
                rows.append({
                    "subj": sid, "cond": cond, "roi": roi,
                    "ap_offset": np.nan, "ap_exponent": np.nan,
                    "peak_cf": np.nan, "peak_pw": np.nan, "peak_bw": np.nan,
                })
            continue

        for roi, idx in roi_idx.items():
            # ROI mean PSD (1d, n_freq)
            psd_1d = np.nanmean(arr[si, idx, :], axis=0)
            #拟合 1/f + peaks
            sm = SpectralModel(
                peak_width_limits=(2, 12),   # 避免窄噪声峰
                max_n_peaks=6,
                min_peak_height=0.1
            )
            sm.fit(freqs, psd_1d, f_fit)

            # 取参数
            ap_offset   = float(sm.get_params("aperiodic", "offset"))
            ap_exponent = float(sm.get_params("aperiodic", "exponent"))
            peaks = sm.get_params("periodic")
            peak_cf, peak_pw, peak_bw = pick_best_peak(peaks, target_band)

            rows.append({
                "subj": sid,
                "cond": cond,
                "roi": roi,
                "ap_offset": ap_offset,
                "ap_exponent": ap_exponent,
                "peak_cf": peak_cf,
                "peak_pw": peak_pw,
                "peak_bw": peak_bw,
            })

#导出结果
out_csv = Path("F:/cue1_epoch/peakdetection_theta_results.csv")
df.to_csv(out_csv, index=False)

In [ ]:
print("npz keys:", d.files)

In [ ]:
#peak detection-cue1-plot


# Cue Sat

In [ ]:
#按Cond Compute power spectrum - cuesat
import re
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path 
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

data_dir = Path("F:/cuesat_epoch")
epo_files = sorted(data_dir.glob("sub_*_cue-epo_cuesat-epo.fif"))
conds = ["cue/ext/rep/sat","cue/ext/swi/sat","cue/int/rep/sat","cue/int/swi/sat"]

#PSD
fmin,fmax=4.0, 30.0
method="welch"
n_fft=256
n_overlap=100

def parse_subj_id(fname: str) -> str:
    m = re.match(r"(sub_[^_]+_\d+_\d+)_cue-epo_cuesat-epo\.fif$", fname)
    return m.group(1) if m else Path(fname).stem

subj_ids = []
tmp_store = {c: [] for c in conds}
freqs_ref, ch_names_ref = None, None

epochs0 = mne.read_epochs(epo_files[0], preload=True, verbose="ERROR").pick("eeg")
ch_names_ref = epochs0.ch_names
n_ch = len(ch_names_ref)

for f in epo_files:
    subj_ids.append(parse_subj_id(f.name))
    epochs = mne.read_epochs(f, preload=True, verbose="ERROR")
    epochs = epochs.copy().pick_types(eeg=True)


    for cond in conds:
        if cond not in epochs.event_id:
            tmp_store[cond].append(None)
            continue

        ep = epochs[cond]
        psd = ep.compute_psd(
            method=method, fmin=fmin, fmax=fmax,
            n_fft=n_fft, n_overlap=n_overlap,
            verbose="ERROR",
        )
        psd_data = psd.get_data()   # (n_trials, n_ch, n_freq)
        freqs = psd.freqs

        if freqs_ref is None:
            freqs_ref = freqs
            n_freq = len(freqs_ref)
        else:
            if len(freqs_ref) != len(freqs) or not np.allclose(freqs_ref, freqs):
                raise ValueError("freqs 不一致：请统一采样率/n_fft/fmin-fmax")

        subj_mean = psd_data.mean(axis=0)  # (n_ch, n_freq)
        tmp_store[cond].append(subj_mean)

# cond -> (n_subj, n_ch, n_freq)
psd_stack = {}
for cond in conds:
    arr = np.full((len(subj_ids), n_ch, len(freqs_ref)), np.nan, dtype=float)
    for i, v in enumerate(tmp_store[cond]):
        if v is not None:
            arr[i] = v
    psd_stack[cond] = arr

#plot
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
axes = axes.ravel()

for ax, cond in zip(axes, conds):
    arr = psd_stack[cond]                       # (n_subj, n_ch, n_freq)
    grand = np.nanmean(arr, axis=(0, 1))        # 平均 subjects + channels -> (n_freq,)
    ax.plot(freqs_ref, grand)
    ax.set_title(cond)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power (V²/Hz)")
    ax.grid(True, alpha=0.3)
    miss = np.isnan(arr).all(axis=(1, 2)).sum()
    if miss > 0:
        ax.text(0.02, 0.95, f"missing subj: {miss}", transform=ax.transAxes, va="top")

plt.tight_layout()
plt.show()

out_path = Path("F:/cuesat_epoch/psd_4conds_subjectlevel_eegonly.npz")

np.savez_compressed(
    out_path,
    subj_ids=np.array(subj_ids, dtype=object),
    ch_names=np.array(ch_names_ref, dtype=object),
    freqs=freqs_ref,
    **psd_stack,
)

print("Saved to:", out_path)

In [ ]:
#分离底噪-sat
import specparam
import numpy as np
import matplotlib.pyplot as plt
from specparam import SpectralModel
import pandas as pd

conds = ["cue/ext/rep/sat","cue/ext/swi/sat","cue/int/rep/sat","cue/int/swi/sat"]
npz_path = Path("F:/cuesat_epoch/psd_4conds_subjectlevel_eegonly.npz")
d = np.load(npz_path, allow_pickle=True)
freqs_ref = d["freqs"]
freqs = freqs_ref
f_fit = (4, 30)

roi_map = {
    "frontal":   ["Fp1","Fp2","AF3","AF4","F7","F3","Fz","F4","F8","FC5","FC1","FC2","FC6"],
    "parietal":  ["CP5","CP1","CP2","CP6","P7","P3","Pz","P4","P8","PO3","PO4"],
    "occipital": ["O1","Oz","O2","PO7","PO8"],
}

roi_idx = {}
for roi_name, chs in roi_map.items():
    idx = [ch_names_ref.index(ch) for ch in chs if ch in ch_names_ref]
    if len(idx) == 0:
        raise ValueError(f"{roi_name} ROI 在 ch_names_ref 里一个都没匹配到，检查通道命名啊！！！！！")
    roi_idx[roi_name] = idx

#specparam
def pick_best_peak(peaks, band):
    #试啊
    for cf_key, pw_key, bw_key in [("cf","pw","bw"), ("center_frequency","power","bandwidth")]:
        try:
            cf = np.array(sm.get_params("periodic", cf_key), dtype=float)
            pw = np.array(sm.get_params("periodic", pw_key), dtype=float)
            bw = np.array(sm.get_params("periodic", bw_key), dtype=float)
            break
        except Exception:
            cf = pw = bw = None
    if cf is None or cf.size == 0:
        return (np.nan, np.nan, np.nan)
    fmin, fmax = band
    mask = (cf >= fmin) & (cf <= fmax)
    if not np.any(mask):
        return (np.nan, np.nan, np.nan)

    sel = np.where(mask)[0][np.argmax(pw[mask])]
    return (cf[sel], pw[sel], bw[sel])

# 需要检测的频段
target_band = (13,28)  

rows = []

for si, sid in enumerate(subj_ids):
    for cond in conds:
        arr = psd_stack[cond]  # (n_subj, n_ch, n_freq)
        if np.isnan(arr[si]).all():
            for roi in roi_idx:
                rows.append({
                    "subj": sid, "cond": cond, "roi": roi,
                    "ap_offset": np.nan, "ap_exponent": np.nan,
                    "peak_cf": np.nan, "peak_pw": np.nan, "peak_bw": np.nan,
                })
            continue

        for roi, idx in roi_idx.items():
            # ROI mean PSD (1d, n_freq)
            psd_1d = np.nanmean(arr[si, idx, :], axis=0)
            #拟合 1/f + peaks
            sm = SpectralModel(
                peak_width_limits=(2, 12),   # 避免窄噪声峰
                max_n_peaks=6,
                min_peak_height=0.1
            )
            sm.fit(freqs, psd_1d, f_fit)

            # 取参数
            ap_offset   = float(sm.get_params("aperiodic", "offset"))
            ap_exponent = float(sm.get_params("aperiodic", "exponent"))
            peaks = sm.get_params("periodic")
            peak_cf, peak_pw, peak_bw = pick_best_peak(peaks, target_band)

            rows.append({
                "subj": sid,
                "cond": cond,
                "roi": roi,
                "ap_offset": ap_offset,
                "ap_exponent": ap_exponent,
                "peak_cf": peak_cf,
                "peak_pw": peak_pw,
                "peak_bw": peak_bw,
            })

out_csv = Path("F:/cuesat_epoch/peakdetection_alphabeta_results_beta.csv")
df.to_csv(out_csv, index=False)

# Target

In [ ]:
#按Cond Compute power spectrum - targ
import re
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path 
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

data_dir = Path("F:/target_epoch")
epo_files = sorted(data_dir.glob("sub_*_targ-epo.fif"))
conds = ["targ/ext/rep/sat","targ/ext/swi/sat","targ/int/rep/sat","targ/int/swi/sat"]

#PSD
fmin,fmax=4.0, 30.0
method="welch"
n_fft=256
n_overlap=100

def parse_subj_id(fname: str) -> str:
    m = re.match(r"(sub_[^_]+_\d+_\d+)_targ-epo.fif$", fname)
    return m.group(1) if m else Path(fname).stem

subj_ids = []
tmp_store = {c: [] for c in conds}
freqs_ref, ch_names_ref = None, None

epochs0 = mne.read_epochs(epo_files[0], preload=True, verbose="ERROR").pick("eeg")
ch_names_ref = epochs0.ch_names
n_ch = len(ch_names_ref)

for f in epo_files:
    subj_ids.append(parse_subj_id(f.name))
    epochs = mne.read_epochs(f, preload=True, verbose="ERROR")
    epochs = epochs.copy().pick_types(eeg=True)


    for cond in conds:
        if cond not in epochs.event_id:
            tmp_store[cond].append(None)
            continue

        ep = epochs[cond]
        psd = ep.compute_psd(
            method=method, fmin=fmin, fmax=fmax,
            n_fft=n_fft, n_overlap=n_overlap,
            verbose="ERROR",
        )
        psd_data = psd.get_data()   # (n_trials, n_ch, n_freq)
        freqs = psd.freqs

        if freqs_ref is None:
            freqs_ref = freqs
            n_freq = len(freqs_ref)
        else:
            if len(freqs_ref) != len(freqs) or not np.allclose(freqs_ref, freqs):
                raise ValueError("freqs 不一致：请统一采样率/n_fft/fmin-fmax")

        subj_mean = psd_data.mean(axis=0)  # (n_ch, n_freq)
        tmp_store[cond].append(subj_mean)

# cond -> (n_subj, n_ch, n_freq)
psd_stack = {}
for cond in conds:
    arr = np.full((len(subj_ids), n_ch, len(freqs_ref)), np.nan, dtype=float)
    for i, v in enumerate(tmp_store[cond]):
        if v is not None:
            arr[i] = v
    psd_stack[cond] = arr

#plot
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
axes = axes.ravel()

for ax, cond in zip(axes, conds):
    arr = psd_stack[cond]                       # (n_subj, n_ch, n_freq)
    grand = np.nanmean(arr, axis=(0, 1))        # 平均 subjects + channels -> (n_freq,)
    ax.plot(freqs_ref, grand)
    ax.set_title(cond)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power (V²/Hz)")
    ax.grid(True, alpha=0.3)
    miss = np.isnan(arr).all(axis=(1, 2)).sum()
    if miss > 0:
        ax.text(0.02, 0.95, f"missing subj: {miss}", transform=ax.transAxes, va="top")

plt.tight_layout()
plt.show()

In [ ]:
out_path = Path("F:/target_epoch/psd_4conds_subjectlevel_eegonly.npz")

np.savez_compressed(
    out_path,
    subj_ids=np.array(subj_ids, dtype=object),
    ch_names=np.array(ch_names_ref, dtype=object),
    freqs=freqs_ref,
    **psd_stack,
)

print("Saved to:", out_path)

In [ ]:
#分离底噪-target
import specparam
import numpy as np
import matplotlib.pyplot as plt
from specparam import SpectralModel
import pandas as pd

conds = ["targ/ext/rep/sat","targ/ext/swi/sat","targ/int/rep/sat","targ/int/swi/sat"]
npz_path = Path("F:/target_epoch/psd_4conds_subjectlevel_eegonly.npz")
d = np.load(npz_path, allow_pickle=True)
freqs_ref = d["freqs"]
freqs = freqs_ref
f_fit = (4, 30)

roi_map = {
    "frontal":   ["Fp1","Fp2","AF3","AF4","F7","F3","Fz","F4","F8","FC5","FC1","FC2","FC6"],
    "parietal":  ["CP5","CP1","CP2","CP6","P7","P3","Pz","P4","P8","PO3","PO4"],
    "occipital": ["O1","Oz","O2","PO7","PO8"],
}

roi_idx = {}
for roi_name, chs in roi_map.items():
    idx = [ch_names_ref.index(ch) for ch in chs if ch in ch_names_ref]
    if len(idx) == 0:
        raise ValueError(f"{roi_name} ROI 在 ch_names_ref 里一个都没匹配到，检查通道命名啊！！！！！")
    roi_idx[roi_name] = idx

#specparam
def pick_best_peak(peaks, band):
    #试啊
    for cf_key, pw_key, bw_key in [("cf","pw","bw"), ("center_frequency","power","bandwidth")]:
        try:
            cf = np.array(sm.get_params("periodic", cf_key), dtype=float)
            pw = np.array(sm.get_params("periodic", pw_key), dtype=float)
            bw = np.array(sm.get_params("periodic", bw_key), dtype=float)
            break
        except Exception:
            cf = pw = bw = None
    if cf is None or cf.size == 0:
        return (np.nan, np.nan, np.nan)
    fmin, fmax = band
    mask = (cf >= fmin) & (cf <= fmax)
    if not np.any(mask):
        return (np.nan, np.nan, np.nan)

    sel = np.where(mask)[0][np.argmax(pw[mask])]
    return (cf[sel], pw[sel], bw[sel])

# 需要检测的频段
target_band = (13,28)  

rows = []

for si, sid in enumerate(subj_ids):
    for cond in conds:
        arr = psd_stack[cond]  # (n_subj, n_ch, n_freq)
        if np.isnan(arr[si]).all():
            for roi in roi_idx:
                rows.append({
                    "subj": sid, "cond": cond, "roi": roi,
                    "ap_offset": np.nan, "ap_exponent": np.nan,
                    "peak_cf": np.nan, "peak_pw": np.nan, "peak_bw": np.nan,
                })
            continue

        for roi, idx in roi_idx.items():
            # ROI mean PSD (1d, n_freq)
            psd_1d = np.nanmean(arr[si, idx, :], axis=0)
            #拟合 1/f + peaks
            sm = SpectralModel(
                peak_width_limits=(2, 12),   # 避免窄噪声峰
                max_n_peaks=6,
                min_peak_height=0.1
            )
            sm.fit(freqs, psd_1d, f_fit)

            # 取参数
            ap_offset   = float(sm.get_params("aperiodic", "offset"))
            ap_exponent = float(sm.get_params("aperiodic", "exponent"))
            peaks = sm.get_params("periodic")
            peak_cf, peak_pw, peak_bw = pick_best_peak(peaks, target_band)

            rows.append({
                "subj": sid,
                "cond": cond,
                "roi": roi,
                "ap_offset": ap_offset,
                "ap_exponent": ap_exponent,
                "peak_cf": peak_cf,
                "peak_pw": peak_pw,
                "peak_bw": peak_bw,
            })

out_csv = Path("F:/target_epoch/peakdetection_alphabeta_results_beta.csv")
df.to_csv(out_csv, index=False)